### This will not include passive OSINT

In [34]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [35]:
df = pd.read_csv("../data/Full_lexical_features.csv") # don't forget this only has 10K data
df.head()

,url,type,url_length,domain,path,domain_length,path_length,num_dots,num_hyphens,num_underscores,...,num_digits,num_letters,num_special_chars,has_ip,has_at_symbol,has_double_slash_redirect,has_https_token_in_domain,has_suspicious_word,num_subdomains,domain_entropy
0,br-icloud.com.br,phishing,16,NaN,br-icloud.com.br,0,16,2,1,0,...,0,13,3,0,0,0,0,0,0,0.000000
1,mp3raid.com/music/krizz_kaliko.html,benign,35,NaN,mp3raid.com/music/krizz_kaliko.html,0,35,2,0,1,...,1,29,5,0,0,0,0,0,0,0.000000
2,bopsecrets.org/rexroth/cr/1.htm,benign,31,NaN,bopsecrets.org/rexroth/cr/1.htm,0,31,2,0,0,...,1,25,5,0,0,0,0,0,0,0.000000
3,http://www.garage-pirenne.be/index.php?option=...,defacement,88,www.garage-pirenne.be,/index.php,21,10,3,1,2,...,7,63,18,0,0,0,0,0,1,3.308751
4,http://adventure-nicaragua.net/index.php?optio...,defacement,235,adventure-nicaragua.net,/index.php,23,10,2,1,1,...,22,199,14,0,0,0,0,0,0,3.501398


In [36]:
print(df['type'].value_counts())

type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64


* phishing --> Fake website pretending to be real
* Benign --> Normal, safe website
* Defacement --> Website that got hacked and altered
* Malware --> Website that infects your device

### URL for phishing URL ->
    https://sahe.in/jir/journal_management/production/plagiarism_files/2paper_12.pdf

In [37]:
print(df.columns)
print(df['type'].value_counts())
print(df['type'].dtype)

Index(['url', 'type', 'url_length', 'domain', 'path', 'domain_length',
       'path_length', 'num_dots', 'num_hyphens', 'num_underscores',
       'num_slashes', 'num_digits', 'num_letters', 'num_special_chars',
       'has_ip', 'has_at_symbol', 'has_double_slash_redirect',
       'has_https_token_in_domain', 'has_suspicious_word', 'num_subdomains',
       'domain_entropy'],
      dtype='str')
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64
str


### Train and test 

In [38]:

X = df.drop(columns=['url', 'domain', 'path', 'type'])
y = df['type']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label mapping:", label_mapping)

#Training 
X_train, X_test, y_train, y_test = train_test_split(X,y_encoded,test_size=0.2,random_state=42,stratify=y_encoded)

Label mapping: {'benign': np.int64(0), 'defacement': np.int64(1), 'malware': np.int64(2), 'phishing': np.int64(3)}


### Logistic Regression

In [39]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=2000,class_weight='balanced',n_jobs=-1)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

print("Logistic Regression Results")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test,y_pred_lr,target_names=le.classes_))

Logistic Regression Results
Accuracy: 0.7545
              precision    recall  f1-score   support

      benign       0.95      0.74      0.83     85621
  defacement       0.78      0.89      0.83     19292
     malware       0.51      0.74      0.60      6504
    phishing       0.40      0.68      0.51     18822

    accuracy                           0.75    130239
   macro avg       0.66      0.76      0.69    130239
weighted avg       0.82      0.75      0.77    130239



### Random Forest

In [40]:

'''
I will first used a randomforest model since it's the data is a int/floats (aka jsut numbers) and see which features
are the main factor to determine would also help alot
'''


rf = RandomForestClassifier(n_estimators=400,random_state=42,n_jobs=-1,class_weight="balanced",max_depth=None)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test,y_pred,target_names=le.classes_))

Accuracy: 0.9414

Classification Report:
              precision    recall  f1-score   support

      benign       0.98      0.95      0.96     85621
  defacement       0.97      0.99      0.98     19292
     malware       0.98      0.94      0.96      6504
    phishing       0.77      0.87      0.82     18822

    accuracy                           0.94    130239
   macro avg       0.92      0.94      0.93    130239
weighted avg       0.95      0.94      0.94    130239



In [41]:

#* Confusion matrix -> measure how well a classification model is performing

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm,index=le.classes_,columns=le.classes_)
print("Confusion Matrix:")
print(cm_df)

Confusion Matrix:
            benign  defacement  malware  phishing
benign       81108          14       43      4456
defacement      42       19059       24       167
malware         40          94     6102       268
phishing      1951         483       53     16335


In [42]:

#* feature importance -> this will be good for reporting 

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

print("Top 15 features:")
print(feature_importance.head(15))

Top 15 features:
                feature  importance
1         domain_length    0.144686
16       domain_entropy    0.139026
6           num_slashes    0.117603
7            num_digits    0.087327
2           path_length    0.079745
15       num_subdomains    0.073208
8           num_letters    0.072159
9     num_special_chars    0.071526
0            url_length    0.059461
3              num_dots    0.047630
4           num_hyphens    0.033598
10               has_ip    0.028076
5       num_underscores    0.023878
14  has_suspicious_word    0.020624
11        has_at_symbol    0.001009


### Graidient Boost

In [43]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=300,learning_rate=0.1,max_depth=3,random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print("Gradient Boosting Results")
print(f"Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}")
print(classification_report(y_test,y_pred_gb,target_names=le.classes_))

Gradient Boosting Results
Accuracy: 0.9298
              precision    recall  f1-score   support

      benign       0.95      0.97      0.96     85621
  defacement       0.93      0.97      0.95     19292
     malware       0.94      0.85      0.90      6504
    phishing       0.82      0.73      0.77     18822

    accuracy                           0.93    130239
   macro avg       0.91      0.88      0.89    130239
weighted avg       0.93      0.93      0.93    130239



### XGBoost

In [44]:
from xgboost import XGBClassifier

xgb = XGBClassifier(objective='multi:softmax',
    num_class=len(le.classes_),
    n_estimators=400,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("XGBoost Results")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(classification_report(y_test,y_pred_xgb,target_names=le.classes_))

XGBoost Results
Accuracy: 0.9453
              precision    recall  f1-score   support

      benign       0.96      0.97      0.97     85621
  defacement       0.96      0.99      0.97     19292
     malware       0.98      0.92      0.95      6504
    phishing       0.85      0.78      0.81     18822

    accuracy                           0.95    130239
   macro avg       0.94      0.92      0.93    130239
weighted avg       0.94      0.95      0.94    130239



### Compare the accuracy score form each model

In [46]:
results = pd.DataFrame({
    "Model": ["Logistic Regression","Random Forest", "Gradient Boosting", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test,y_pred),
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_xgb)
    ]
})

print(results)

                 Model  Accuracy
0  Logistic Regression  0.754459
1        Random Forest  0.941377
2    Gradient Boosting  0.929798
3              XGBoost  0.945262


### Compare Macro F1 score
Treats all classes equally, regardless of their frequency. Calculates the average F1 score (balance of precision and recall) independently for each class and then takes the unweighted mean

In [ ]:
from sklearn.metrics import f1_score

'''
i feel that deciding by macro F1 might be best option since we should take the matrics and compare them equally, rather than
expecting them to all have the same amount of data
'''
print("Macro F1 Scores")
print("Logistic:", f1_score(y_test, y_pred_lr, average='macro'))
print("Random Forest:", f1_score(y_test, y_pred, average='macro'))
print("Gradient Boosting:", f1_score(y_test, y_pred_gb, average='macro'))
print("XGBoost:", f1_score(y_test, y_pred_xgb, average='macro'))

Macro F1 Scores
Logistic: 0.6931532757869531
Random Forest: 0.9287011652421703
Gradient Boosting: 0.8938824418354087
XGBoost: 0.9250672103514691


### Picking between the two modles

In [ ]:
print("=========================  XGBoost =========================")
print(classification_report(y_test,y_pred_xgb,target_names=le.classes_))

print("=========================  Random Forest ====================")
print(classification_report(y_test,y_pred,target_names=le.classes_))

=========================  XGBoost =========================
              precision    recall  f1-score   support

      benign       0.96      0.97      0.97     85621
  defacement       0.96      0.99      0.97     19292
     malware       0.98      0.92      0.95      6504
    phishing       0.85      0.78      0.81     18822

    accuracy                           0.95    130239
   macro avg       0.94      0.92      0.93    130239
weighted avg       0.94      0.95      0.94    130239

=========================  Random Forest ====================
              precision    recall  f1-score   support

      benign       0.98      0.95      0.96     85621
  defacement       0.97      0.99      0.98     19292
     malware       0.98      0.94      0.96      6504
    phishing       0.77      0.87      0.82     18822

    accuracy                           0.94    130239
   macro avg       0.92      0.94      0.93    130239
weighted avg       0.95      0.94      0.94    130239



### Conclusion 
- RF finds more phishing URLs (higher recall)
- XGBoost is more conservative (higher precision)
- We will piorities Recall over percision with the logic of security